# This is the python interface:



In [ ]:
##load the library
import ipywidgets as widgets
from IPython.display import display, HTML
import serial
import keyboard
import threading
import time
import numpy as np

In [2]:
##main class, store port information, method to command on the other project
class interface():
    def __init__(self):
        print(f'interaction tool is ready')
        self.port = '' ## the port name for connect8ion
        self.out = widgets.Output(layout={'border': '1px solid black', 'overflow_y':'scroll'}) ## machine output region
        self.rate =115200
        self.read_signal = True
        self._input_signal = True
        self.user_input = widgets.Output(layout={'border': '1px solid black'}) ## user input history
        self.user_input.append_stdout('Welcome! click the submit ang type to submit to processor'+'\n')
        self.command_his = '' ## user current input
        self.input_text = widgets.Text(description="Command:") ## input text widget
        self.connect_thread = threading.Thread()
        self.ser =serial.Serial() ## this is to set the port
        self.commandlib = {
            'default':'1\r0\r0\r10000\r0\r0\r1\r0\r62500\r\r',
            'Fpga_low_noise': '3\r1\r1\r0\r0\r10000\r0\r0\r1\r0\r62500\r\r',
            'Fpga_normal' : '3\r0\r1\r0\r0\r10000\r0\r0\r1\r100\r300\r\r',
            'return begining' : '1\r0\r0\r10000\r0\r1\r\r' +'\r\r\rq',
            'thresold_3mv':'1\r2\r3\r-3\r0\r10000\r0\r0\r1\r0\r62500\r\r' #x/2**13
                          
                          }
    def _keyboard_detect(self):
        keyboard.add_hotkey('space', lambda: print('space was pressed'))
        keyboard.wait()
    def set_port_rate(self,P_and_I):
        self.port = P_and_I[0]
        self.rate = P_and_I[1]
    ## this is to show the user history input
    def set_input_signal(self):
        self._input_signal =True
        if(len(self.input_text.value)!=0 and self.input_text.value[0] =='q'):
            self.command_his = self.input_text.value   
        else:
            self.command_his = self.input_text.value.replace('\\','r') +'\r'
        self.user_input.append_stdout(self.command_his+'\n')
    
    def command(self,input_num,_command):
        self._input_signal =True
        if(_command == 'WindowSize'): # change the window size
            self.command_his = '1\r0\r0\r'+str(input_num)+'\r0\r0\r1\r0\r62.5\r\r'
        elif (_command == 'CrossSize'): # this is to change the cross number
            self.command_his = '1\r0\r1\r'+str(input_num)+'\r0\r0\r1\r100\r300\r\r'
        elif(_command == 'testFpgaMode'): # this can be done to change the test mode
            input_num *= 8.192 # 2^14/2000
            self.command_his = '2\r'+str(input_num)+'\r'
        elif(_command == 'changeThreshold'): # this can change the threshold
            self.command_his = '1\r2\r'+str(input_num[0])+'\r'+str(input_num[1])+'\r0\r'+input_num[2]+'\r0\r0\r1\r100\r300\r\r'
        elif(_command =='errormode'):
            self.command_his = '1\r0\r1\r'+1250+'\r0\r1\r1\r100\r300\r\r'
        else:
            self.command_his = _command
        self.user_input.append_stdout(self.command_his+'\n')
        self.write_input()
    ## this is to display the port config block
    def show_out_config(self,display_list):    
        with self.out:
            display(widgets.VBox(display_list))
    ## this is to show the input state
    def show_input_state(self):
        with self.user_input:
            try:
                display(widgets.VBox([self.input_text,submit_button_input]))
            except:
                print(f'some components of widget are undefined')
    def display(self):
        display(widgets.HBox([self.out,self.user_input]))
    def clear(self):
        self.out.clear_output()
        self.user_input.clear_output()
    def connect_i(self):
        self.ser =serial.Serial(self.port, self.rate,timeout = 0.1)
        while self.read_signal:
            try:
                message = self.ser.readlines()
            
                #print(message.decode('utf-8'))
                for i in message:
                    self.out.append_stdout(i.decode('utf-8'))
            except UnicodeDecodeError :
                print("utf-8 decoding may be wrong or the port is not open:the original text")
                print(message)
            '''
            try:
                message = self.ser.read(500)
                #print(message.decode('utf-8'))
                message = message.decode('utf-8')
                message.replace('\r','')
                self.out.append_stdout(message)
            except:
                print("not connected")
            '''

    def write_input(self):
        if(self._input_signal):
            if(not self.ser.is_open):
                self.ser.open()
                print(f'port: config {self.ser.is_open} to Ture')
            print(f'the command is : {repr(self.command_his)}')
            a = self.ser.write(self.command_his.encode('utf-8'))
            self.command_his = ''
            self.input_text.value = ''
            self.input_signal = False
        else:
            print(f'the input signal is not submitted')

In [3]:
## the user interface for port input
port_input = widgets.Text(
    value = '/dev/tty.usbserial-DP03QLIM', #/dev/tty.usbserial-DM01X6D1 #/dev/tty.usbserial-DK0FY4LE
    placeholder='Type something',
    description='Port_name:',
    disabled=False
)
rate = widgets.Text(
    value = '115200',
    placeholder='Type something',
    description='rate:',
    disabled=False
)
submit_button = widgets.Button( description='submit',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='submit',
    icon='check',  # (FontAwesome names without the `fa-` prefix)
    layout=widgets.Layout(width='50%', height='100%')
)


i = interface()

    
def observe_fuc(b):
    
    i.set_port_rate([port_input.value,rate.value])
    if(i.ser.is_open):
        print(f'observe the device')
submit_button.on_click(observe_fuc)

i.show_out_config([port_input,rate,submit_button])
i.display()

interaction tool is ready


In [4]:
##clear out the input
i.out.outputs = []
i.port
threading.enumerate()


[<_MainThread(MainThread, started 140704520467264)>,
 <Thread(IOPub, started daemon 123145366671360)>,
 <Heartbeat(Heartbeat, started daemon 123145383460864)>,
 <Thread(Thread-3 (_watch_pipe_fd), started daemon 123145401323520)>,
 <Thread(Thread-4 (_watch_pipe_fd), started daemon 123145418113024)>,
 <ControlThread(Control, started daemon 123145434902528)>,
 <HistorySavingThread(IPythonHistorySavingThread, started 123145451692032)>,
 <ParentPollerUnix(Thread-2, started daemon 123145469018112)>]

In [5]:
## set up the thread to connect the port
#"i.connect_thread.join()
i.connect_thread = threading.Thread(target = i.connect_i)
i.connect_thread.start()
threading.enumerate()

[<_MainThread(MainThread, started 140704520467264)>,
 <Thread(IOPub, started daemon 123145366671360)>,
 <Heartbeat(Heartbeat, started daemon 123145383460864)>,
 <Thread(Thread-3 (_watch_pipe_fd), started daemon 123145401323520)>,
 <Thread(Thread-4 (_watch_pipe_fd), started daemon 123145418113024)>,
 <ControlThread(Control, started daemon 123145434902528)>,
 <HistorySavingThread(IPythonHistorySavingThread, started 123145451692032)>,
 <ParentPollerUnix(Thread-2, started daemon 123145469018112)>,
 <Thread(Thread-6 (connect_i), started 123145485807616)>]

In [6]:
#listen the port
submit_button_input = widgets.Button( description='submit',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='submit',
    icon='check',  # (FontAwesome names without the `fa-` prefix)
    layout=widgets.Layout(width='50%', height='100%')
)
#ver 1 detection
'''
def detect():
    while(True):
        i._keyboard_detect()
        i.set_input_signal()
        i.write_input()
keyboard_detect = threading.Thread(target = detect)
keyboard_detect.start() 
'''


#ver 2 detection
def submit_input(b):
    print(f'summitted')
    i.set_input_signal()
    i.write_input()
submit_button_input.on_click(submit_input)
print(f'show the command : {repr(i.command)}')

i.show_input_state()
i.display()

show the command : <bound method interface.command of <__main__.interface object at 0x7fd1f6a7cac0>>


In [ ]:
i.command(0,'1\r2\r3\r-3\r0\r10000\r0\r\r1\r0\r10000\r\r')

In [ ]:
i.command(0,'q')

In [ ]:
i.command(0,'1\r0\r0\r10000\r0\r0\r1\r0\r62500\r\r')

In [ ]:
i.command(0,i.commandlib['default']) #thresold_3mv

In [ ]:

i.command(20000,'WindowSize')
#i.command(0,i.commandlib['Fpga_low_noise'])